# Merged Preprocessing: Lindsey (TikTok) + Karin (Pantip)

**Inputs**:
- Lindsey: `gpt_label.csv` (comment IDs + text), `blank.csv` (comment metadata)
- Karin: `00_Datasets.txt` (Pantip post-level data)

**Outputs**:
- `merged_post_level.csv` — unified post-level dataframe with `source` column
- (also intermediate per-source CSVs for inspection)

**Pipeline**:
1. Lindsey side: apply `analyze_comment()` rules → join with metadata (keep `amp_prob` continuous, no binarization)
2. Karin side: parse IP-sharing lists → compute `sockpuppet_group_id` → filter `tag ⊃ 'การเมือง'`
3. Harmonize column names and types
4. Concatenate with `source` discriminator
5. Export

**Label semantics** (deliberately kept separate):
- TikTok rows: `amp_prob` continuous in [0, 1], `is_sockpuppet` = NaN
- Pantip rows: `is_sockpuppet` binary in {0, 1}, `amp_prob` = NaN

In [1]:
import re
import ast
import numpy as np
import pandas as pd
import networkx as nx
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Adjust these paths to match your local layout
# TIKTOK_LABEL_CSV   = 'Lindsey_dataset/gpt_label.csv'
TIKTOK_LABEL_CSV   = 'tiktok_labeled.csv'
TIKTOK_META_CSV    = 'Lindsey_dataset/blank.csv'
PANTIP_RAW_TXT     = 'Karin_dataset/00_Datasets.txt'
OUT_MERGED         = 'merged_post_level.csv'
OUT_TIKTOK         = 'tiktok_preprocessed.csv'
OUT_PANTIP         = 'pantip_preprocessed.csv'

POLITICAL_TAG       = 'การเมือง'

## 1. Lindsey (TikTok) preprocessing

Steps:
1. Load `gpt_label.csv` and `blank.csv`, join on `cid`
2. Apply `analyze_comment()` to fill `amp_prob` (the raw file has it blank)
3. Keep `amp_prob` **continuous** — no binarization
4. Standardize column names to merged schema (set `is_sockpuppet` = NaN since TikTok has no IP-based label)

In [2]:
def analyze_comment(text):
    """Lindsey's hand-crafted rule-based amp_prob assignment (verbatim from labeling.ipynb)."""
    if not text or not isinstance(text, str):
        return 0.0
    text = text.strip()
    lower = text.lower()
    emoji_pat = r'[\U0001F600-\U0001F64F]|[\U0001F300-\U0001F5FF]|[\U0001F680-\U0001F6FF]|[\U0001F1E0-\U0001F1FF]|[\U00002600-\U000026FF]|[\U00002700-\U000027BF]|❤️|💚|✌️|🥰|😭|😢|😊|😍|🔥|👏|🙏|💪|😘|🤗|🙌'
    emojis = re.findall(emoji_pat, text)
    ec = len(emojis)
    txt_clean = re.sub(emoji_pat, '', text)
    txt_clean = re.sub(r'[!@#$%^&*()_+\-=\[\]{};\':"\\|,.<>\?]', ' ', txt_clean).strip()
    words = [w for w in txt_clean.split() if w]
    wc = len(words)
    if ec > 9: return 1.0
    if 7 <= ec <= 9 and wc <= 1: return 1.0
    for t in ['lugaw','meryenda','pinklawan','rappler']:
        if t in lower: return 1.0
    if '@bbm' in lower or '@bongbongmarcos' in lower: return 1.0
    if 'leni use pink flag' in lower and 'bbm use philippines flag' in lower: return 1.0
    for t in ['deserves more views','dapat trending','bakit hindi pa viral','push mo to','fyp','for you page','algorithm','viral','trending','blow this up']:
        if t in lower: return 1.0
    for t in ['facebook','fb page','twitter','instagram','youtube','tara sa fb']:
        if t in lower: return 1.0
    if re.match(r'^(bbm|sara|sarah|marcos|bongbong)\s*[❤️💚✌️🥰😭😢😊😍🔥👏🙏💪😘🤗🙌!]*\s*$', text, re.IGNORECASE): return 0.9
    if wc <= 2:
        for p in ['this!','true!','solid','unity','panalo','yes!','oo!','tama!']:
            if p in lower: return 0.9
    for k in ['nakakakilabot','nakakaiyak','nakaka-proud','chills','shivers','grabe yung','goosebumps','kakaiyak','naiiyak','literally cried','nakakaproud','grabe','nakakatakot']:
        if k in lower: return 0.8
    for p in ['first time ko mag-comment','silent supporter ako pero','hindi ako political pero','first time voter ako','never ko naging paboritong','hindi ako fan pero']:
        if p in lower: return 0.8
    for loc in ['manila','cebu','davao','quezon city','makati','taguig','pasig','marikina','caloocan','las pinas','muntinlupa','paranaque','pasay','luzon','visayas','mindanao','bicol','ilocos','pangasinan','cavite','laguna','batangas','rizal','bulacan','pampanga','zambales','baguio']:
        if loc in lower: return 0.8
    for a in ['deserve','proud of you','country deserves him','deserve mo yan','proud ako','deserve niya','deserve natin']:
        if a in lower: return 0.8
    if 4 <= ec <= 6 and wc <= 4: return 0.67
    for s in ['best president ever','greatest leader of all time','perfect candidate','flawless victory','best president','greatest leader','perfect na','the best talaga','walang kapantay']:
        if s in lower: return 0.7
    for p in ['tara na mga','calling all','mga ka-','present mga solid','mga solid','team bbm','mga bbm']:
        if p in lower: return 0.3
    if wc == 0 and 0 < ec <= 4: return 0.3
    if wc > 8 and ec <= 3: return 0.1
    if 5 <= wc <= 8 and ec <= 2: return 0.15
    return 0.2

print('Loading TikTok data...')
tt_label = pd.read_csv(TIKTOK_LABEL_CSV)
tt_meta  = pd.read_csv(TIKTOK_META_CSV)
print(f'  gpt_label.csv: {len(tt_label):,} rows')
print(f'  blank.csv:     {len(tt_meta):,} rows')

# Join (cid is the natural key)
tt = pd.merge(tt_meta, tt_label[['cid','text']], on='cid', how='inner', suffixes=('','_lbl'))
# Prefer text from blank.csv (full text with emojis), but fall back if missing
tt['text'] = tt['text'].fillna(tt.get('text_lbl', ''))
if 'text_lbl' in tt.columns:
    tt = tt.drop(columns=['text_lbl'])
print(f'  Joined: {len(tt):,} comments')

# Apply analyze_comment to (re)generate amp_prob
print('Applying analyze_comment to all texts...')
tt['amp_prob'] = tt['text'].apply(analyze_comment)
print(f'  amp_prob distribution:')
print(tt['amp_prob'].value_counts().sort_index(ascending=False).to_string())
print(f'\nKept continuous (no binarization). Range: [{tt["amp_prob"].min():.2f}, {tt["amp_prob"].max():.2f}]')

Loading TikTok data...
  gpt_label.csv: 19,842 rows
  blank.csv:     19,842 rows
  Joined: 19,842 comments
Applying analyze_comment to all texts...
  amp_prob distribution:
amp_prob
1.00    1205
0.90     558
0.80    1525
0.70      21
0.67    1189
0.30     415
0.20    7151
0.15    3004
0.10    4774

Kept continuous (no binarization). Range: [0.10, 1.00]


In [3]:
# Standardize to merged schema
tt_clean = pd.DataFrame({
    'source':              'tiktok',
    'post_id':             tt['cid'].astype(str),
    'container_id':        tt['aweme_id'].astype(str),
    'user_id':             'tiktok_' + tt['user_uid'].astype(str),
    'datetime':            pd.to_datetime(tt['create_time'], unit='s'),
    'text':                tt['text'].fillna('').astype(str),
    'length':              tt['text'].fillna('').str.len(),
    'likes_received':      tt['digg_count'].fillna(0).astype(int),
    'replies_received':    tt['reply_comment_total'].fillna(0).astype(int),
    'amp_prob':            tt['amp_prob'],          # continuous, native to TikTok
    'is_sockpuppet':       np.nan,                  # not available on TikTok
    # 'sockpuppet_group_id': -1,                      # Lindsey has no group structure
    # 'tag':                 [[] for _ in range(len(tt))],  # TikTok has no tags
})

tt_clean.to_csv(OUT_TIKTOK, index=False, encoding='utf-8-sig')
print(f'Saved {OUT_TIKTOK}: {tt_clean.shape}')
print(f'  Unique users:  {tt_clean["user_id"].nunique():,}')
print(f'  Unique videos: {tt_clean["container_id"].nunique():,}')
print(f'  amp_prob ≥ 0.8 rows: {(tt_clean["amp_prob"] >= 0.8).sum():,}  (high-confidence amplifier signal)')
tt_clean.head(3)

Saved tiktok_preprocessed.csv: (19842, 11)
  Unique users:  18,454
  Unique videos: 36
  amp_prob ≥ 0.8 rows: 3,288  (high-confidence amplifier signal)


,source,post_id,container_id,user_id,datetime,text,length,likes_received,replies_received,amp_prob,is_sockpuppet
0,tiktok,7094446713754714881,7093346652542569755,tiktok_6911212009079342081,2022-05-06 02:36:52,fun fact\nLeni use pink flag\nbbm use Philippi...,52,13,3,1.0,NaN
1,tiktok,7094399216679961345,7094264285508963611,tiktok_6801708565356479490,2022-05-05 23:32:34,yesss louderrrrr❤💚,18,5,0,0.2,NaN
2,tiktok,7094371390119805697,7094264285508963611,tiktok_61546214063681536,2022-05-05 21:44:38,myyy heart huhu literally cried :(,34,9,0,0.8,NaN


## 2. Karin (Pantip) preprocessing

Steps:
1. Load `00_Datasets.txt`, parse `user_sameIP` lists
2. Filter to `IP_check == True` (only users whose IPs are traceable can be labeled)
3. Build sockpuppet groups via connected components on IP-sharing graph
4. Compute `is_sockpuppet` (= 1 if `user_sameIP` is non-empty)
5. **Filter to political threads** (`tag` contains 'การเมือง')
6. Standardize to merged schema

In [4]:
print('Loading Pantip data...')
# 00_Datasets.txt is JSON, not CSV. Handle BOM + invalid control chars + possible truncation.
import json as _json
with open(PANTIP_RAW_TXT, 'r', encoding='utf-8-sig') as f:
    raw = f.read().strip()
raw = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', raw)
if raw.endswith(','):
    raw = raw[:-1] + '\n]'
elif not raw.endswith(']'):
    raw = raw + '\n]'
data = _json.loads(raw, strict=False)
p = pd.DataFrame(data)
print(f'  Loaded: {p.shape}')
print(f'  Columns: {list(p.columns)}')

# Parse list-like columns
def parse_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        s = x.strip()
        if s == '' or s == '[]':
            return []
        if s.startswith('['):
            try:
                return ast.literal_eval(s)
            except Exception:
                return []
    return []

for col in ['user_sameIP', 'tag', 'profile_like']:
    if col in p.columns:
        p[col] = p[col].apply(parse_list)

# Filter to users with traceable IPs
if 'IP_check' in p.columns:
    p = p[p['IP_check'] == True].copy()
    print(f'  After IP_check filter: {p.shape}')

# Binary label
p['is_sockpuppet'] = p['user_sameIP'].apply(lambda x: 1 if (isinstance(x, list) and len(x) > 0) else 0)
print(f'  Sockpuppet rows (post-level): {p["is_sockpuppet"].sum():,}')

Loading Pantip data...
  Loaded: (95904, 21)
  Columns: ['num_0', 'num_1', 'num_r', 'thread_id', 'thread_name', 'user_name', 'user_id', 'messages', 'length', 'helpful', 'forum', 'tag', 'datetime', 'profile_like', 'emo_like', 'user_type', 'user_sameIP', 'IP', 'IP_check', 'link', 'messages_ref']
  After IP_check filter: (34960, 21)
  Sockpuppet rows (post-level): 12,263


In [5]:
# Build sockpuppet groups via connected components on user_sameIP
print('Computing sockpuppet groups (connected components on IP-sharing graph)...')
G = nx.Graph()
sock_df = p[p['is_sockpuppet'] == 1].drop_duplicates('user_id')
for _, row in sock_df.iterrows():
    uid = row['user_id']
    G.add_node(uid)
    same_ip = row['user_sameIP'] if isinstance(row['user_sameIP'], list) else []
    for other in same_ip:
        G.add_edge(uid, other)

user_to_group = {}
for gid, component in enumerate(nx.connected_components(G)):
    for u in component:
        user_to_group[u] = gid

# p['sockpuppet_group_id'] = p['user_id'].map(user_to_group).fillna(-1).astype(int) # uncomment if bring back 'sockpuppet_group_id' column
n_groups = len(set(v for v in user_to_group.values()))
print(f'  Sockpuppet groups: {n_groups}')
print(f'  Group size distribution:')
group_sizes = pd.Series([sum(1 for v in user_to_group.values() if v == g) for g in range(n_groups)])
print(group_sizes.describe().to_string())

Computing sockpuppet groups (connected components on IP-sharing graph)...
  Sockpuppet groups: 323
  Group size distribution:
count    323.000000
mean       3.690402
std        6.832749
min        2.000000
25%        2.000000
50%        2.000000
75%        3.000000
max       91.000000


In [6]:
# Filter to political threads
print(f'Filtering to threads where tag contains "{POLITICAL_TAG}"...')
p_pol = p[p['tag'].apply(lambda x: POLITICAL_TAG in x if isinstance(x, list) else False)].copy()
print(f'  Before: {len(p):,} rows, {p["user_id"].nunique():,} users')
print(f'  After:  {len(p_pol):,} rows, {p_pol["user_id"].nunique():,} users')
print(f'  Sockpuppet rows after filter: {p_pol["is_sockpuppet"].sum():,}')
print(f'  Sockpuppet users after filter: {p_pol[p_pol["is_sockpuppet"]==1]["user_id"].nunique():,}')
print(f'  Normal users after filter:    {p_pol[p_pol["is_sockpuppet"]==0]["user_id"].nunique():,}')

Filtering to threads where tag contains "การเมือง"...


  Before: 34,960 rows, 4,895 users
  After:  5,457 rows, 394 users
  Sockpuppet rows after filter: 2,390
  Sockpuppet users after filter: 142
  Normal users after filter:    252


In [7]:
# Parse datetime
p_pol['datetime_parsed'] = pd.to_datetime(p_pol['datetime'], format='%m/%d/%Y %H:%M:%S', errors='coerce')
# Some rows may use a different format — fallback
missing = p_pol['datetime_parsed'].isna()
if missing.any():
    p_pol.loc[missing, 'datetime_parsed'] = pd.to_datetime(p_pol.loc[missing, 'datetime'], errors='coerce')
print(f'Datetime parse: {p_pol["datetime_parsed"].notna().sum():,} / {len(p_pol):,} parsed')

# Reactions count (likes_received analog)
p_pol['likes_received'] = p_pol['profile_like'].apply(lambda x: len(x) if isinstance(x, list) else 0)

# Concatenate title + post text where available
if 'topic' in p_pol.columns and 'messages' in p_pol.columns:
    p_pol['text_full'] = p_pol['topic'].fillna('').astype(str) + ' ' + p_pol['messages'].fillna('').astype(str)
    p_pol['text_full'] = p_pol['text_full'].str.strip()
else:
    p_pol['text_full'] = p_pol.get('messages', pd.Series([''] * len(p_pol))).fillna('').astype(str)

# Standardize to merged schema
p_clean = pd.DataFrame({
    'source':              'pantip',
    'post_id':             p_pol.index.astype(str),                        # no native post id; use row index
    'container_id':        p_pol['thread_id'].astype(str),
    'user_id':             'pantip_' + p_pol['user_id'].astype(str),
    'datetime':            p_pol['datetime_parsed'],
    'text':                p_pol['text_full'],
    'length':              p_pol['text_full'].str.len(),
    'likes_received':      p_pol['likes_received'].astype(int),
    'replies_received':    np.nan,                                          # Pantip lacks a direct analog
    'amp_prob':            np.nan,                                          # Pantip has no rule-based score
    'is_sockpuppet':       p_pol['is_sockpuppet'].astype(int),              # binary, native to Pantip
    # 'sockpuppet_group_id': p_pol['sockpuppet_group_id'].astype(int),
    # 'tag':                 p_pol['tag'],
}).reset_index(drop=True)

p_clean.to_csv(OUT_PANTIP, index=False, encoding='utf-8-sig')
print(f'Saved {OUT_PANTIP}: {p_clean.shape}')
p_clean.head(3)

Datetime parse: 5,457 / 5,457 parsed
Saved pantip_preprocessed.csv: (5457, 11)


,source,post_id,container_id,user_id,datetime,text,length,likes_received,replies_received,amp_prob,is_sockpuppet
0,pantip,0,42111000,pantip_7464651,2023-07-10 09:19:04,ก้าวไกลคือแบบอย่างการเมืองสุจริต ไม่ซื้อเสียงจ...,154,3,NaN,NaN,1
1,pantip,2,42111000,pantip_7576977,2023-07-10 09:35:04,ขึ้น ค่าแรง 450 ทันที สูงอายุ 3000 ติดเตียง 10...,119,2,NaN,NaN,1
2,pantip,7,42111000,pantip_7450328,2023-07-10 13:36:40,ปชป.แมงสาบ 2,12,0,NaN,NaN,1


## 3. Concatenate into unified dataset

In [8]:
merged = pd.concat([tt_clean, p_clean], ignore_index=True)

# Sanity: every row has a source and a user_id; labels are source-specific
assert merged['source'].notna().all()
assert merged['user_id'].notna().all()
# Every row should have exactly one of the two labels populated
tt_mask = merged['source'] == 'tiktok'
p_mask  = merged['source'] == 'pantip'
assert merged.loc[tt_mask, 'amp_prob'].notna().all(),       'TikTok rows missing amp_prob'
assert merged.loc[tt_mask, 'is_sockpuppet'].isna().all(),   'TikTok rows should not have is_sockpuppet'
assert merged.loc[p_mask, 'is_sockpuppet'].notna().all(),   'Pantip rows missing is_sockpuppet'
assert merged.loc[p_mask, 'amp_prob'].isna().all(),         'Pantip rows should not have amp_prob'

print(f'Merged shape: {merged.shape}')
print(f'\nBy source:')
summary = []
for src in ['tiktok', 'pantip']:
    sub = merged[merged['source'] == src]
    if src == 'tiktok':
        # For TikTok, report high-confidence positives (amp_prob >= 0.8)
        user_max = sub.groupby('user_id')['amp_prob'].max()
        summary.append({
            'source': 'tiktok', 'posts': len(sub), 'users': sub['user_id'].nunique(),
            'pos_users_amp>=0.8': int((user_max >= 0.8).sum()),
            'pos_users_amp>0.5':  int((user_max >  0.5).sum()),
        })
    else:
        user_max = sub.groupby('user_id')['is_sockpuppet'].max()
        summary.append({
            'source': 'pantip', 'posts': len(sub), 'users': sub['user_id'].nunique(),
            'sockpuppet_users':   int(user_max.sum()),
            'normal_users':       int((user_max == 0).sum()),
        })
print(pd.DataFrame(summary).to_string(index=False))

# Save
merged.to_csv(OUT_MERGED, index=False, encoding='utf-8-sig')
print(f'\nSaved {OUT_MERGED}: {merged.shape}')

Merged shape: (25299, 11)

By source:
source  posts  users  pos_users_amp>=0.8  pos_users_amp>0.5  sockpuppet_users  normal_users
tiktok  19842  18454              3189.0             4322.0               NaN           NaN
pantip   5457    394                 NaN                NaN             142.0         252.0

Saved merged_post_level.csv: (25299, 11)


## 4. Verification

In [9]:
print('--- Schema check ---')
print(merged.dtypes.to_string())

print('\n--- Missing values (expected: amp_prob NaN on Pantip rows, is_sockpuppet NaN on TikTok rows) ---')
print(merged.isna().sum().to_string())

print('\n--- User ID collision check (should be 0) ---')
tt_uids = set(merged[merged['source']=='tiktok']['user_id'])
p_uids  = set(merged[merged['source']=='pantip']['user_id'])
print(f'Overlap: {len(tt_uids & p_uids)}')

print('\n--- TikTok user-level amp_prob distribution ---')
tt_user_max = merged[merged['source']=='tiktok'].groupby('user_id')['amp_prob'].max()
print(tt_user_max.describe().to_string())
print(f'  Users with max amp_prob >= 0.8: {(tt_user_max >= 0.8).sum():,}')
print(f'  Users with max amp_prob >  0.5: {(tt_user_max >  0.5).sum():,}')
print(f'  Users with max amp_prob <= 0.2: {(tt_user_max <= 0.2).sum():,}')

print('\n--- Pantip user-level is_sockpuppet distribution ---')
p_user_max = merged[merged['source']=='pantip'].groupby('user_id')['is_sockpuppet'].max()
print(p_user_max.value_counts().to_string())

--- Schema check ---
source                      object
post_id                     object
container_id                object
user_id                     object
datetime            datetime64[ns]
text                        object
length                       int64
likes_received               int64
replies_received           float64
amp_prob                   float64
is_sockpuppet              float64

--- Missing values (expected: amp_prob NaN on Pantip rows, is_sockpuppet NaN on TikTok rows) ---
source                  0
post_id                 0
container_id            0
user_id                 0
datetime                0
text                    0
length                  0
likes_received          0
replies_received     5457
amp_prob             5457
is_sockpuppet       19842

--- User ID collision check (should be 0) ---
Overlap: 0

--- TikTok user-level amp_prob distribution ---
count    18454.000000
mean         0.319329
std          0.292846
min          0.100000
25%          0.